# Практика · Чому текст складніший за таблицю> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · Домашнє завдання: [homework.html](homework.html)⏱ **Заміряно: близько 30 секунд** на чотирьох ядрах без відеокарти.Найдовше працюють клітинки з ростом словника — вони кілька разів проходятьувесь корпус у різному порядку.Тут ми **перевіряємо числами** те, що лекція стверджує словами:1. беремо справжній український корпус — переклади інтерфейсів у системі;2. будуємо генератор оголошень і показуємо, що він **не** є текстом;3. міряємо закон Ципфа двома способами й дивимось, чому вони розходяться;4. рахуємо частку словоформ, що трапились рівно раз;5. міряємо, скільки слів треба, щоб покрити 50, 90 і 99 відсотків тексту;6. міряємо закон Гіпса на трьох зернах — окремо для української й англійської;7. знаходимо неоднозначні слова **в корпусі**, а не вигадуємо їх;8. показуємо, що порядок слів міняє зміст, а мішок слів цього не бачить;9. міряємо, як спадає частка невідомих слів, коли корпус росте у 88 разів.

## 0 · СередовищеДрукуємо версії — щоб пізніше було зрозуміло, у чому саме рахувалось.

In [ ]:
import sys, re, glob, gettext, random, time, math
from collections import Counter, defaultdict
import numpy as np

print("Python     ", sys.version.split()[0])
print("NumPy      ", np.__version__)
import sklearn
print("scikit-learn", sklearn.__version__)

## 1 · Корпус: українські переклади в системіНам потрібен **справжній** український текст: не вигаданий, не згенерований і непереписаний. Найдешевше джерело такого тексту вже лежить на диску — це перекладиінтерфейсів. Кожна встановлена програма несе файл `.mo` з парами«англійський оригінал → український переклад».Це дає рідкісну річ: **паралельний корпус**. Два тексти, які означають те саме,але написані різними мовами. Далі ми на цьому зіграємо.⚠️ Української локалі на твоїй машині може не бути. Тому нижче стоїть запобіжник:якщо корпус не набрався, вмикається вбудований мінікорпус зі 105 пар просто в цьомузошиті. Числа тоді будуть інші — і клітинка прямо скаже, який шлях спрацював.

In [ ]:
# 105 пар «оригінал ||| переклад» на випадок, якщо української локалі немає.
# Цього замало для законів Ципфа й Гіпса, але досить, щоб зошит виконався весь.
MINI_CORPUS = """
Failed to open the configuration file ||| Не вдалося відкрити файл налаштувань
The connection was closed by the remote host ||| Зʼєднання розірвано віддаленим вузлом
Cannot convert an associative array to an indexed one ||| Неможливо перетворити асоціативний масив на індексований
Cannot convert an indexed array to an associative one ||| Неможливо перетворити індексований масив на асоціативний
Permission denied while writing to the directory ||| Немає прав на запис до каталогу
The device is busy and cannot be unmounted ||| Пристрій зайнятий і його неможливо відмонтувати
Unknown option in the command line ||| Невідомий параметр у рядку команди
Please choose a different name for the new file ||| Виберіть іншу назву для нового файла
The archive appears to be damaged ||| Схоже, архів пошкоджено
Reading the package list from the repository ||| Читання списку пакунків зі сховища
The certificate has expired three days ago ||| Строк дії сертифіката вичерпано три дні тому
Do you really want to remove the selected items ||| Ви справді хочете вилучити позначені обʼєкти
Saving changes to the current document ||| Збереження змін до поточного документа
The window cannot be resized below the minimum ||| Вікно не можна зменшити нижче за мінімальний розмір
Searching for available network connections ||| Пошук доступних мережевих зʼєднань
The password you entered is too short ||| Введений вами пароль надто короткий
Installation finished with several warnings ||| Встановлення завершено з кількома попередженнями
A newer version of this program is available ||| Доступна новіша версія цієї програми
The disk does not have enough free space ||| На диску недостатньо вільного місця
Copying files to the temporary directory ||| Копіювання файлів до тимчасового каталогу
This action cannot be undone afterwards ||| Цю дію не можна буде скасувати згодом
The server returned an unexpected response ||| Сервер повернув неочікувану відповідь
Waiting for the background task to finish ||| Очікування завершення фонового завдання
Two processes are using the same port ||| Два процеси використовують той самий порт
The image format is not supported by this build ||| Формат зображення не підтримується цією збіркою
Rename the folder and try again ||| Перейменуйте теку і спробуйте ще раз
The list of recent documents has been cleared ||| Список недавніх документів очищено
Some settings will apply after a restart ||| Деякі налаштування застосуються після перезапуску
The printer is offline or out of paper ||| Принтер вимкнено або у ньому скінчився папір
Checking the integrity of the downloaded file ||| Перевірка цілісності звантаженого файла
The keyboard layout has been switched ||| Розкладку клавіатури перемкнено
An update is being prepared in the background ||| Оновлення готується у фоновому режимі
Nothing was found for your search query ||| За вашим пошуковим запитом нічого не знайдено
Select a folder where the report will be saved ||| Виберіть теку, у якій буде збережено звіт
The value must be a positive whole number ||| Значення має бути додатним цілим числом
Reverting the last change in the editor ||| Скасування останньої зміни у редакторі
The clipboard does not contain any text ||| У буфері обміну немає жодного тексту
Adding a new user to the group ||| Додавання нового користувача до групи
The battery level is critically low ||| Рівень заряду батареї критично низький
Sound output device has been changed ||| Пристрій виведення звуку змінено
Sorting the table by the second column ||| Впорядкування таблиці за другим стовпчиком
The table already contains a row with this key ||| Таблиця вже містить рядок із цим ключем
Loading the module from an external library ||| Завантаження модуля із зовнішньої бібліотеки
Building the index of the local collection ||| Побудова покажчика локальної збірки
The camera is used by another application ||| Камеру використовує інша програма
Downloading translations for your language ||| Звантаження перекладів для вашої мови
The screen brightness cannot be adjusted here ||| Яскравість екрана не можна змінити тут
Backup created successfully in the home folder ||| Резервну копію успішно створено у домашній теці
The line contains an unexpected character ||| Рядок містить неочікуваний символ
Compiling the source with optimisation enabled ||| Компіляція коду з увімкненою оптимізацією
The mail message could not be delivered ||| Поштове повідомлення не вдалося доставити
Enter the address of the remote repository ||| Введіть адресу віддаленого сховища
Two files with the same name were found ||| Знайдено два файли з однаковою назвою
Removing an empty directory from the tree ||| Вилучення порожнього каталогу з дерева
The video stream has stopped unexpectedly ||| Відеопотік несподівано зупинився
Changing the owner of the selected file ||| Зміна власника позначеного файла
The application has recovered from a crash ||| Програма відновилася після аварійного завершення
Automatic updates are currently disabled ||| Автоматичне оновлення зараз вимкнено
The requested page could not be rendered ||| Запитану сторінку не вдалося показати
Please confirm the operation with your password ||| Підтвердьте операцію своїм паролем
This key is already bound to another action ||| Цю клавішу вже призначено на іншу дію
Statistics are collected once per hour ||| Статистика збирається раз на годину
The font could not be loaded from the system ||| Шрифт не вдалося завантажити з системи
Merging two branches of the repository ||| Злиття двох гілок сховища
The task queue is empty at the moment ||| Черга завдань наразі порожня
Restoring the previous session windows ||| Відновлення вікон попереднього сеансу
The header of the file is malformed ||| Заголовок файла має неправильний формат
Sharing the folder with other users ||| Оприлюднення теки для інших користувачів
The operation timed out after ten seconds ||| Час очікування операції вичерпано за десять секунд
Type a few letters to filter the list ||| Наберіть кілька літер, щоб відфільтрувати список
The archive contains files outside the target ||| Архів містить файли поза цільовою текою
Preparing the disk for the new partition ||| Підготовка диска до нового розділу
The service will start at the next boot ||| Службу буде запущено при наступному завантаженні
Ignoring an unknown field in the record ||| Пропускання невідомого поля у записі
The chosen colour scheme is too dark ||| Вибрана схема кольорів надто темна
Playback has been paused by the system ||| Відтворення призупинено системою
The name of the variable must not be empty ||| Назва змінної не може бути порожньою
Reading the manual page for this command ||| Читання сторінки посібника для цієї команди
Two devices are competing for the same address ||| Два пристрої змагаються за ту саму адресу
The database was locked by another process ||| Базу даних заблоковано іншим процесом
Exporting the report to a separate file ||| Експортування звіту до окремого файла
The label text is longer than the field ||| Текст мітки довший за поле
Counting the words in the current selection ||| Підрахунок слів у поточному виділенні
The tree view could not be expanded fully ||| Дерево не вдалося розгорнути повністю
Please wait while the cache is rebuilt ||| Зачекайте, доки буде перебудовано кеш
Notifications from this program are muted ||| Сповіщення від цієї програми вимкнено
The signature of the package does not match ||| Підпис пакунка не збігається
Opening the link in a new browser window ||| Відкриття посилання у новому вікні браузера
The list of installed extensions is empty ||| Список встановлених розширень порожній
Applying the filter to every selected row ||| Застосування фільтра до кожного позначеного рядка
The value was rounded to two decimal places ||| Значення округлено до двох знаків після коми
Synchronising the local copy with the server ||| Синхронізація локальної копії із сервером
The input file and the output file are the same ||| Вхідний файл і вихідний файл збігаються
Show the hidden entries in this directory ||| Показати приховані записи у цьому каталозі
The process was killed by an external signal ||| Процес завершено зовнішнім сигналом
Choosing the language of the interface ||| Вибір мови інтерфейсу
The report will be sent to the address below ||| Звіт буде надіслано на адресу нижче
Recording has been stopped by the user ||| Запис зупинено користувачем
The margin cannot exceed the page width ||| Поле не може перевищувати ширину сторінки
Verifying the checksum of every downloaded block ||| Перевірка контрольної суми кожного звантаженого блоку
The history of commands has been saved ||| Історію команд збережено
Attaching a comment to the selected line ||| Долучення коментаря до позначеного рядка
The tool needs administrator rights to continue ||| Засобу потрібні права адміністратора, щоб продовжити
Two windows are showing the same document ||| Два вікна показують той самий документ
The animation speed was reduced automatically ||| Швидкість анімації автоматично зменшено
"""


def load_corpus(min_len=30):
    """Повертає список трійок (програма, англійський оригінал, український переклад).

    Довші за min_len переклади беремо тому, що короткі рядки на кшталт «Гаразд»
    або «%s» — це не речення, а підписи кнопок, і вони перекосили б статистику.
    """
    docs = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
        except Exception:
            continue                      # пошкоджений або чужого формату файл просто пропускаємо
        program = path.split('/')[-1][:-3]
        for source, target in catalog._catalog.items():
            if (isinstance(source, str) and isinstance(target, str)
                    and len(target) > min_len and 'Project-Id' not in target):
                docs.append((program, source, target))
    return docs


t_start = time.time()
docs = load_corpus()
CORPUS_SOURCE = 'локаль системи'

if len(docs) < 2000:                      # локалі немає або вона майже порожня
    CORPUS_SOURCE = 'вбудований мінікорпус'
    docs = []
    for line in MINI_CORPUS.strip().split('\n'):
        source, target = line.split('|||')
        docs.append(('mini', source.strip(), target.strip()))

programs = sorted({program for program, _, _ in docs})
print("джерело корпусу:", CORPUS_SOURCE)
print("документів:     ", len(docs))
print("програм:        ", len(programs))
print("завантаження:   ", round(time.time() - t_start, 1), "с")

## 2 · Що це за текст насправдіПерш ніж робити з корпусу висновки, треба чесно сказати, **який він**. Подивимосьна три випадкові документи.

In [ ]:
sample = random.Random(0).sample(docs, 3)
for program, source, target in sample:
    print(f"[{program}]")
    print("  англ:", source[:90])
    print("  укр: ", target[:90])

Видно одразу: це **вузький домен**. Технічна лексика, короткі речення, багатонаказового способу й повідомлень про помилки. Робити з такого корпусу висновкипро українську мову взагалі не можна — і ми не будемо. Але для того, що насцікавить у цій темі (як розподілені частоти й чи насичується словник), доменне має значення: закони Ципфа й Гіпса виконуються в будь-якому справжньому тексті.## 3 · Найпростіша токенізаціяЩоб рахувати слова, треба спершу вирішити, **що таке слово**. Найпростіше:послідовність літер української абетки плюс апостроф. Для англійської — латинськілітери.Це рішення грубе, і тема 02 присвячена саме тому, наскільки воно грубе. Тут вононам годиться: усі порівняння в цьому зошиті зроблені **одним і тим самим** способом,тож систематична похибка однакова скрізь і висновків не міняє.

In [ ]:
# літери української абетки плюс апостроф ʼ (U+02BC) — він усередині слова: зʼєднання
UA_WORD = re.compile(r"[абвгґдежзиійклмнопрстуфхцчшщьюяєіїʼ]+")
EN_WORD = re.compile(r"[a-z]+")


def ua_tokens(text):
    """Ріже український рядок на слова. Регістр прибираємо, щоб «Файл» і «файл»
    рахувались одним словом — інакше словник роздувся б удвічі ні за що."""
    return UA_WORD.findall(text.lower())


def en_tokens(text):
    return EN_WORD.findall(text.lower())


print(ua_tokens("Не вдалося відкрити зʼєднання з сервером «локальний-1»."))
print(en_tokens("Failed to open the connection to the server."))

## 4 · Перший замір: скільки в корпусі слів і скільки різнихДва різні числа, які легко переплутати:* **слововживання** — скільки слів у тексті всього, разом із повторами;* **словоформи** — скільки серед них **різних** рядків.

In [ ]:
ua_stream, en_stream = [], []
for program, source, target in docs:
    ua_stream.extend(ua_tokens(target))
    en_stream.extend(en_tokens(source))

ua_freq = Counter(ua_stream)
en_freq = Counter(en_stream)

ua_chars = sum(len(word) for word in ua_stream)
en_chars = sum(len(word) for word in en_stream)

print(f"українська:  {len(ua_stream):>8} слововживань, {len(ua_freq):>6} словоформ")
print(f"англійська:  {len(en_stream):>8} слововживань, {len(en_freq):>6} словоформ")
print()
print(f"середня довжина слова: укр {ua_chars/len(ua_stream):.3f} літери, "
      f"англ {en_chars/len(en_stream):.3f}")
print(f"літер усього:          укр {ua_chars}, англ {en_chars} "
      f"(відношення {ua_chars/en_chars:.3f})")

Зверни увагу на пару чисел, яка й буде головним українським сюжетом теми:**той самий зміст** записано меншою кількістю слів українською, але **більшою**кількістю різних словоформ. Одне й те саме поняття українська розкладає на форми(файл, файла, файлу, файли, файлів), а англійська обходиться однією.## 5 · Спроба, яка провалилась: генератор оголошеньПерш ніж вірити корпусу, треба перевірити протилежне: а чи не можна взяти текст**із таблиці**? Напишемо генератор оголошень про продаж авто — чотири шаблони,десять слотів, кілька десятків значень у кожному. Виглядає як текст. Чи є він текстом?

In [ ]:
BRAND = ["тойота", "хонда", "мазда", "рено", "шкода", "ніссан", "форд", "опель"]
BODY  = ["седан", "універсал", "хетчбек", "кросовер", "мінівен"]
FUEL  = ["бензин", "дизель", "газ", "гібрид", "електро"]
STATE = ["ідеальний", "добрий", "задовільний", "новий", "чудовий"]
COLOR = ["чорний", "білий", "сірий", "синій", "червоний", "зелений", "сріблястий"]
TOWN  = ["київ", "львів", "одеса", "харків", "дніпро", "полтава", "вінниця", "суми"]
EXTRA = ["кондиціонер", "підігрів сидінь", "камера заднього виду", "круїз контроль",
         "шкіряний салон", "навігація", "датчик дощу", "зимова гума"]
TAIL  = ["торг можливий", "обмін не цікавить", "розгляну варіанти", "терміново",
         "власник один", "документи в порядку", "дзвоніть у будь який час"]

TEMPLATES = [
 "продам автомобіль {b} {body}, {y} рік випуску, пробіг {km} тисяч кілометрів. "
 "паливо {fuel}, колір {c}. стан {st}. є {e1} та {e2}. місто {t}. "
 "ціна {p} тисяч гривень. {tail}.",
 "терміново продається {b}, кузов {body}, {y} року. двигун на {fuel}. "
 "пробіг {km} тисяч. колір {c}, стан {st}. додатково {e1}. {t}. "
 "{p} тисяч гривень, {tail}.",
 "{b} {body} {y} року у місті {t}. пробіг {km} тисяч кілометрів, паливо {fuel}, "
 "колір {c}. технічний стан {st}. комплектація містить {e1}, {e2}. "
 "вартість {p} тисяч гривень. {tail}.",
 "автомобіль {b}, тип кузова {body}, рік {y}. колір {c}, пробіг {km} тисяч. "
 "{fuel}. стан {st}, є {e2}. розташування {t}. ціна {p} тисяч. {tail}.",
]


def make_ad(rnd):
    """Одне оголошення. Кожен слот — незалежний випадковий вибір зі свого списку."""
    return rnd.choice(TEMPLATES).format(
        b=rnd.choice(BRAND), body=rnd.choice(BODY), y=rnd.randint(2005, 2022),
        km=rnd.randint(10, 300), fuel=rnd.choice(FUEL), c=rnd.choice(COLOR),
        st=rnd.choice(STATE), e1=rnd.choice(EXTRA), e2=rnd.choice(EXTRA),
        t=rnd.choice(TOWN), p=rnd.randint(80, 900), tail=rnd.choice(TAIL))


print(make_ad(random.Random(0)))

Тепер міряємо головне: **чи росте словник**, коли оголошень стає більше.

In [ ]:
def grid_upto(preset, total, minimum=3):
    """Обрізає сітку під фактичний розмір даних.

    Потрібне для мінікорпусу: жорстко зашиті 20 000 документів там просто
    не існують, і всі заміри нижче впали б на виході за межі масиву.
    """
    marks = [n for n in preset if n < total]
    if len(marks) < minimum:
        marks = sorted({max(1, int(total * part)) for part in (0.05, 0.1, 0.25, 0.5)})
        marks = [n for n in marks if n < total]
    return marks + [total]


DOC_GRID = grid_upto([10, 20, 50, 100, 200, 500, 1000, 2000, 5000,
                      10000, 20000, 50000], len(docs))


def vocab_growth_by_docs(texts_for_seed, grid):
    """Скільки різних слів набралось після перших n документів — для трьох зерен."""
    table = {}
    for seed in (0, 1, 2):
        texts = texts_for_seed(seed)
        seen, i, row = set(), 0, []
        for n in grid:
            while i < n and i < len(texts):
                seen.update(ua_tokens(texts[i]))
                i += 1
            row.append(len(seen))
        table[seed] = row
    return table


def corpus_texts(seed):
    order = list(docs)
    random.Random(seed).shuffle(order)
    return [target for _, _, target in order]


def generated_texts(seed):
    rnd = random.Random(seed)
    return [make_ad(rnd) for _ in range(max(DOC_GRID))]


growth_corpus = vocab_growth_by_docs(corpus_texts, DOC_GRID)
growth_gen = vocab_growth_by_docs(generated_texts, DOC_GRID)

print("документів   словник генератора   словник корпусу   (середнє з трьох зерен,")
print("                     розкид               розкид      розкид = max - min)")
for j, n in enumerate(DOC_GRID):
    g = [growth_gen[s][j] for s in (0, 1, 2)]
    c = [growth_corpus[s][j] for s in (0, 1, 2)]
    print(f"{n:>10}   {int(round(np.mean(g))):>10} ± {max(g)-min(g):<5}"
          f"   {int(round(np.mean(c))):>10} ± {max(c)-min(c):<5}")

Словник **насичується на сотні оголошень** і далі не росте зовсім. Четверта тисячаоголошень не приносить жодного нового слова — бо приносити нема звідки: усі словавже перелічені у списках вище.Це і є діагноз. Наш «текст» — це таблиця, вбрана в речення. У таблиці стовпчик«колір» має сім можливих значень, і восьмого не буде ніколи. Шаблонний генераторуспадкував цю властивість цілком.Перевіримо ще три ознаки, і всі три — на **трьох зернах**, щоб не переплутативластивість генератора з випадковістю одного прогону.

In [ ]:
def zipf_slope(freqs, points=50):
    """Нахил прямої у координатах «логарифм рангу — логарифм частоти».

    Ранги беремо не всі підряд, а рівномірно по декадах: інакше десятки тисяч
    рідкісних слів переважать сотню частих, і пряма опише хвіст, а не закон.
    """
    values = np.array(sorted(freqs, reverse=True), dtype=float)
    n = len(values)
    idx = np.unique(np.round(np.logspace(0, np.log10(n), points)).astype(int)) - 1
    ranks = np.arange(1, n + 1)
    slope, _ = np.polyfit(np.log(ranks[idx]), np.log(values[idx]), 1)
    return slope


def heaps_beta(stream, points=26):
    """Показник β у V ≈ k·N^β: наскільки швидко росте словник із довжиною тексту."""
    marks = np.unique(np.round(np.logspace(2, np.log10(len(stream)), points)).astype(int))
    seen, curve, j = set(), [], 0
    for i, word in enumerate(stream, 1):
        seen.add(word)
        while j < len(marks) and i == marks[j]:
            curve.append((i, len(seen)))
            j += 1
    xs = np.log([p[0] for p in curve])
    ys = np.log([p[1] for p in curve])
    beta, log_k = np.polyfit(xs, ys, 1)
    return beta, math.exp(log_k), curve


gen_betas, gen_vocab = [], []
print("зерно   словник   нахил Ципфа   бета Гіпса   слів рівно раз")
for seed in (0, 1, 2):
    rnd = random.Random(seed)
    stream = []
    while len(stream) < 101000:
        stream.extend(ua_tokens(make_ad(rnd)))
    stream = stream[:101000]
    freq = Counter(stream)
    beta, _, _ = heaps_beta(stream)
    hapax = sum(1 for count in freq.values() if count == 1)
    gen_betas.append(beta)
    gen_vocab.append(len(freq))
    print(f"{seed:>5}   {len(freq):>7}   {zipf_slope(freq.values()):>11.4f}   "
          f"{beta:>10.4f}   {hapax:>14}")

print(f"\nсереднє по трьох зернах: β = {np.mean(gen_betas):.4f}, "
      f"словник = {int(np.mean(gen_vocab))} слів")

Три зерна дають той самий вирок: словник рівно сто слів, показник Гіпса біля нуля(словник не росте), **жодного** слова, що трапилось один раз. У справжньому текстівсі три числа інші, і зараз ми це побачимо.## 6 · Закон Ципфа на справжньому корпусіВідсортуємо всі словоформи за спаданням частоти. Місце в цьому списку зветься**рангом**: ранг 1 — найчастіше слово, ранг 2 — друге за частотою і так далі.

In [ ]:
ranked = ua_freq.most_common()
print(f"словоформ у корпусі: {len(ranked)}\n")
print("ранг   слово               частота")
for rank in (1, 2, 3, 4, 5, 10, 50, 100, 500, 1000, 5000, 10000, 20000, len(ranked)):
    if rank <= len(ranked):
        word, count = ranked[rank - 1]
        print(f"{rank:>5}   {word:<18}  {count:>7}")

Найчастіше слово трапляється в сотні разів частіше за тисячне. Це вже не схожена таблицю, де стовпчики більш-менш рівноправні.Тепер порахуємо **нахил** — наскільки швидко частота спадає з рангом. Ципф каже,що в логарифмічних координатах точки лягають на пряму з нахилом близько −1.І тут є пастка, про яку рідко говорять: **нахил залежить від того, як його рахувати**.

In [ ]:
values = np.array([count for _, count in ranked], dtype=float)
ranks = np.arange(1, len(values) + 1)

def slope_all(limit):
    """Найпростіша підгонка: усі ранги від 1 до limit, кожен з однаковою вагою."""
    return np.polyfit(np.log(ranks[:limit]), np.log(values[:limit]), 1)[0]

def slope_decades(limit, points=50):
    """Підгонка по декадах: беремо 50 рангів, рівномірно розкиданих по логарифму."""
    idx = np.unique(np.round(np.logspace(0, np.log10(limit), points)).astype(int)) - 1
    return np.polyfit(np.log(ranks[idx]), np.log(values[idx]), 1)[0]

ZIPF_WINDOWS = grid_upto([100, 500, 1000, 5000, 10000], len(values))
print("до рангу    усі ранги    по декадах")
for limit in ZIPF_WINDOWS:
    print(f"{limit:>8}    {slope_all(limit):>9.4f}    {slope_decades(limit):>10.4f}")

Два стовпчики розходяться, і що ширше вікно рангів — то сильніше. На повномудіапазоні «усі ранги» дає близько −1.45, а «по декадах» — майже точно −1.Причина проста й важлива. Рангів із частотою 1 у корпусі десятки тисяч, а рангівіз частотою понад тисячу — кілька десятків. Якщо кожна точка має однакову вагу,пряму тягне за собою плаский хвіст, а не сам закон. Підгонка по декадах даєкожному порядку величини однакову вагу — і тоді видно те, що й мав на увазі Ципф.Тримай це в голові: **число, яке залежить від способу підрахунку, треба називатиразом зі способом**. Далі в курсі так буде постійно.## 7 · Третина словника трапилась рівно разСлово, що трапилось у корпусі рівно один раз, англійською зветься *hapax legomenon*(«сказане одного разу»). Порахуємо, скільки їх.

In [ ]:
spectrum = Counter(ua_freq.values())

print("трапилось   таких        частка     частка    накопичено   накопичено")
print("   разів   словоформ   словника    тексту   словника, %   тексту, %")
cum_types = cum_tokens = 0
for k in range(1, 11):   # у мінікорпусі частина рядків буде нульова — це нормально
    types_k = spectrum.get(k, 0)
    cum_types += types_k
    cum_tokens += k * types_k
    print(f"{k:>8}   {types_k:>9}   {100*types_k/len(ua_freq):>8.2f}%  "
          f"{100*k*types_k/len(ua_stream):>8.2f}%   {100*cum_types/len(ua_freq):>10.2f}   "
          f"{100*cum_tokens/len(ua_stream):>9.2f}")

hapax_ua = spectrum.get(1, 0)
hapax_en = sum(1 for count in en_freq.values() if count == 1)
rare = sum(spectrum.get(k, 0) for k in (1, 2, 3))
print()
print(f"словоформ рівно раз: укр {hapax_ua} ({100*hapax_ua/len(ua_freq):.2f}% словника), "
      f"англ {hapax_en} ({100*hapax_en/len(en_freq):.2f}%)")
print(f"словоформ до трьох разів включно: {rare} "
      f"({100*rare/len(ua_freq):.2f}% словника, "
      f"{100*sum(k*spectrum.get(k,0) for k in (1,2,3))/len(ua_stream):.2f}% тексту)")

Ось перший наслідок, який ламає табличну звичку. Понад третина словника — це слова,про які ми маємо **одне спостереження**. Ні середнього, ні розкиду, ні надійноїстатистики. У таблиці таке трапляється з рідкісною категорією; у тексті це**норма**, і жоден обсяг даних її не скасовує — далі побачимо чому.## 8 · Скільки стовпчиків мала б таблицяУяви, що ти таки хочеш покласти текст у таблицю: одне слово — один стовпчик.Скільки стовпчиків треба, щоб покрити задану частку тексту?

In [ ]:
cumulative = np.cumsum(values) / values.sum()

print("топ-K слів   покриває тексту")
COVER_GRID = grid_upto([10, 50, 100, 250, 500, 1000, 2500, 5000, 10000, 20000],
                       len(values))
for k in COVER_GRID:
    print(f"{k:>10}   {100*cumulative[k-1]:>14.2f}%")

print()
for target in (0.50, 0.80, 0.90, 0.95, 0.99):
    need = int(np.searchsorted(cumulative, target) + 1)
    print(f"щоб покрити {100*target:>5.1f}% тексту, треба {need:>6} найчастіших слів")

Двісті пʼятдесят дев'ять слів дають половину тексту — і ці слова майже нічого неозначають («не», «для», «у», «з»). А щоб дійти до 99 відсотків, стовпчиків требадвадцять тисяч. Кожен новий відсоток покриття коштує дедалі дорожче: це той самийхвіст Ципфа, побачений з іншого боку.## 9 · Словник не насичується: закон ГіпсаНайважливіший замір теми. Візьмемо корпус, перемішаємо документи й подивимось, якросте кількість **різних** слів у міру того, як ми читаємо текст.Емпіричний закон Гіпса каже: V ≈ k·N<sup>β</sup>, де N — скільки слів прочитано,V — скільки різних серед них, а β — показник між нулем і одиницею. β = 0 означало б«словник не росте зовсім» (наш генератор), β = 1 означало б «кожне слово нове».Перемішування — це і є місце, де ховається випадковість, тому **три зерна**.

In [ ]:
def shuffled_streams(seed):
    """Той самий корпус у випадковому порядку документів — окремо укр і англ потоки."""
    order = list(docs)
    random.Random(seed).shuffle(order)
    ua, en = [], []
    for program, source, target in order:
        ua.extend(ua_tokens(target))
        en.extend(en_tokens(source))
    return ua, en


t_start = time.time()
beta_ua, beta_en = [], []
heaps_curve_ua = None
for seed in (0, 1, 2):
    ua, en = shuffled_streams(seed)
    b_ua, k_ua, curve = heaps_beta(ua)
    b_en, k_en, _ = heaps_beta(en)
    beta_ua.append(b_ua)
    beta_en.append(b_en)
    if seed == 0:
        heaps_curve_ua = curve
        k_first = k_ua
    print(f"зерно {seed}:  укр β = {b_ua:.4f} (k = {k_ua:.3f})   "
          f"англ β = {b_en:.4f} (k = {k_en:.3f})")

print()
print(f"укр:  середнє β = {np.mean(beta_ua):.4f}, розкид = {max(beta_ua)-min(beta_ua):.4f}")
print(f"англ: середнє β = {np.mean(beta_en):.4f}, розкид = {max(beta_en)-min(beta_en):.4f}")
print(f"різниця β = {np.mean(beta_ua)-np.mean(beta_en):.4f}, "
      f"тобто у {(np.mean(beta_ua)-np.mean(beta_en))/max(max(beta_ua)-min(beta_ua), 1e-9):.1f} раза "
      f"більша за розкид української")
print("рахувалось", round(time.time() - t_start, 1), "с")

**Різниця більша за розкид — отже, вона справжня.** Це те саме правило, з яким тижив у машинному навчанні: різниця, менша за розкид між зернами, різницею не є.Тепер подивимось на саму криву й на те, що вона обіцяє.

In [ ]:
TOKEN_GRID = grid_upto([100, 300, 1000, 3000, 10000, 30000, 100000, 300000],
                       len(ua_stream))


def vocab_at(stream, grid):
    """Розмір словника після перших n слів потоку — для кожного n із сітки."""
    seen, i, row = set(), 0, []
    for n in grid:
        while i < n and i < len(stream):
            seen.add(stream[i])
            i += 1
        row.append(len(seen))
    return row


gen_stream = []
gen_rnd = random.Random(0)
while len(gen_stream) < len(ua_stream):
    gen_stream.extend(ua_tokens(make_ad(gen_rnd)))
gen_stream = gen_stream[:len(ua_stream)]

ua_shuffled, en_shuffled = shuffled_streams(0)
row_ua = vocab_at(ua_shuffled, TOKEN_GRID)
row_en = vocab_at(en_shuffled, TOKEN_GRID)
row_gen = vocab_at(gen_stream, TOKEN_GRID)

print("прочитано слів   укр   англ   генератор")
for j, n in enumerate(TOKEN_GRID):
    print(f"{n:>14}   {row_ua[j]:>5}   {row_en[j]:>5}   {row_gen[j]:>7}")

# β на однаковій довжині тексту для всіх трьох потоків — щоб порівняння було чесним
same_length = len(ua_stream)
for name, stream in (("укр", ua_shuffled), ("англ", en_shuffled[:same_length]),
                     ("генератор", gen_stream)):
    beta, k, _ = heaps_beta(stream)
    print(f"β на {same_length} словах, {name:<10} {beta:>7.4f}  (k = {k:.3f})")
print()

print("прочитано слів   різних слів   нових слів на наступну тисячу")
for i in range(1, len(heaps_curve_ua)):
    n, v = heaps_curve_ua[i]
    previous_n, previous_v = heaps_curve_ua[i - 1]
    per_1000 = 1000 * (v - previous_v) / (n - previous_n)
    print(f"{n:>14}   {v:>11}   {per_1000:>28.1f}")

beta_mean = float(np.mean(beta_ua))
print()
print(f"за формулою V ≈ k·N^β із β = {beta_mean:.4f} і k = {k_first:.3f}:")
for n in (10**6, 10**7, 10**8):
    print(f"  {n:>12} слів  ->  {k_first * n**beta_mean:>12.0f} різних словоформ")

Крива не виходить на полицю. Кожна наступна тисяча слів приносить менше новихсловоформ, ніж попередня, — але **більше за нуль**, і так до нескінченності.Формально: похідна спадає, але залишається додатною, бо β менше одиниці й більше нуля.Це і є головна відмінність від таблиці. Стовпчик «колір» має скінченну множинузначень. Стовпчик «текст» — не має.## 10 · Українська: більше форм на менше слівТепер зіграємо на тому, що корпус паралельний. Два боки означають **те саме**,тож будь-яка різниця між ними — це різниця мов, а не різниця змісту.

In [ ]:
print(f"{'':<22}{'українська':>12}{'англійська':>13}")
print(f"{'слововживань':<22}{len(ua_stream):>12}{len(en_stream):>13}")
print(f"{'словоформ':<22}{len(ua_freq):>12}{len(en_freq):>13}")
print(f"{'літер':<22}{ua_chars:>12}{en_chars:>13}")
print(f"{'середнє слово, літер':<22}{ua_chars/len(ua_stream):>12.3f}{en_chars/len(en_stream):>13.3f}")
print(f"{'показник Гіпса β':<22}{np.mean(beta_ua):>12.4f}{np.mean(beta_en):>13.4f}")
print()
print("відношення укр / англ:")
print(f"  слововживань {len(ua_stream)/len(en_stream):.3f}   літер {ua_chars/en_chars:.3f}   "
      f"словоформ {len(ua_freq)/len(en_freq):.3f}   бета {np.mean(beta_ua)/np.mean(beta_en):.3f}")
print()
print(f"слів українською менше у {len(en_stream)/len(ua_stream):.3f} раза")
print(f"словоформ українською більше у {len(ua_freq)/len(en_freq):.3f} раза")
print(f"літер українською більше у {ua_chars/en_chars:.3f} раза")

Три числа разом і дають портрет морфологічно багатої мови:* слів **менше** — українська обходиться без артиклів і частини службових слів;* літер **більше** — самі слова довші;* різних словоформ **більше в півтора раза** — бо одне поняття дає багато форм.І показник Гіпса вищий саме через третє. Це не «українська складніша». Цеконкретна, вимірна властивість: на однаковій кількості прочитаних слів модельукраїнської мусить тримати більший словник. Що з цим робити — тема[03 «Морфологія»](../03-morphology/lecture.html), а як це бʼє по токенізації —тема [02](../02-tokenization/lecture.html).## 11 · Неоднозначність: знайдемо її в корпусі, а не вигадаємоПаралельний корпус дає ще один подарунок. Візьмемо всі записи, де **з обох боківрівно одне слово** — це фактично двомовний словничок, зібраний із реальних програм.Тоді питання «скільки значень має слово» стає вимірним: скільки різних англійськихслів перекладено тим самим українським.

In [ ]:
en_to_ua = defaultdict(Counter)
ua_to_en = defaultdict(Counter)
single_pairs = 0

for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
    try:
        with open(path, 'rb') as f:
            catalog = gettext.GNUTranslations(f)
    except Exception:
        continue
    for source, target in catalog._catalog.items():
        if not (isinstance(source, str) and isinstance(target, str)):
            continue
        left, right = en_tokens(source), ua_tokens(target)
        # беремо лише пари «одне слово -> одне слово»: це і є словникова стаття
        if len(left) == 1 and len(right) == 1 and len(left[0]) > 2 and len(right[0]) > 2:
            en_to_ua[left[0]][right[0]] += 1
            ua_to_en[right[0]][left[0]] += 1
            single_pairs += 1

if single_pairs == 0:      # мінікорпус: збираємо те саме з наявних пар
    for program, source, target in docs:
        left, right = en_tokens(source), ua_tokens(target)
        if len(left) == 1 and len(right) == 1:
            en_to_ua[left[0]][right[0]] += 1
            ua_to_en[right[0]][left[0]] += 1
            single_pairs += 1

print(f"однослівних пар: {single_pairs}")
print(f"різних англійських слів: {len(en_to_ua)}, українських: {len(ua_to_en)}")
if single_pairs == 0:
    print("у мінікорпусі однослівних пар немає — цей замір потребує повної локалі")
if en_to_ua:
    multi_en = sum(1 for v in en_to_ua.values() if len(v) > 1)
    multi_ua = sum(1 for v in ua_to_en.values() if len(v) > 1)
    print(f"англійських слів із більш ніж одним перекладом: {multi_en} "
          f"({100*multi_en/len(en_to_ua):.2f}%)")
    print(f"українських слів із більш ніж одним джерелом:    {multi_ua} "
          f"({100*multi_ua/len(ua_to_en):.2f}%)")

In [ ]:
if not ua_to_en:
    print("нічого показати: словничок порожній (див. клітинку вище)")

print("українське слово   ->   англійські слова, які ним перекладено")
top_ua = sorted(ua_to_en.items(), key=lambda kv: -len(kv[1]))[:8]
for word, sources in top_ua:
    names = ", ".join(name for name, _ in sources.most_common(8))
    print(f"  {word:<14} {len(sources)}: {names}")

print()
print("англійське слово   ->   українські слова, якими його перекладено")
top_en = sorted(en_to_ua.items(), key=lambda kv: -len(kv[1]))[:6]
for word, targets in top_en:
    names = ", ".join(name for name, _ in targets.most_common(8))
    print(f"  {word:<14} {len(targets)}: {names}")

Подивись на слово **«запис»**. Воно стоїть у корпусі перекладом і для `entry`(рядок у списку), і для `write` (дія запису на диск), і для `record` (структураданих), і для `login` (вхід у систему). Це не помилка перекладачів — цебагатозначність, і вона в мові скрізь.Для моделі, яка бачить лише рядок символів, «запис» у двох різних реченнях —**одне й те саме**. Розвʼязувати цю проблему курс буде довго: почне з контекстув блоці про вектори слів і закінчить у трансформері.## 12 · Порядок слів не декораціяПошукаємо в корпусі пари документів, які складаються **з тих самих слів**, алев різному порядку. Якщо порядок не важив би, таких пар не мало б існувати —або вони означали б те саме.

In [ ]:
by_bag = defaultdict(set)
for program, source, target in docs:
    words = ua_tokens(target)
    if 3 <= len(words) <= 9:                 # короткі речення, щоб пару було видно оком
        by_bag[tuple(sorted(words))].add(tuple(words))

collisions = {bag: orders for bag, orders in by_bag.items() if len(orders) > 1}
print(f"груп із однаковим складом слів: {len(by_bag)}")
print(f"з них таких, де порядок різний: {len(collisions)}")

In [ ]:
# найпоказовіша пара з корпусу: ті самі шість слів, протилежний зміст
sentence_a = "неможливо перетворити асоціативний масив на індексований"
sentence_b = "неможливо перетворити індексований масив на асоціативний"
words_a, words_b = sentence_a.split(), sentence_b.split()

print("A:", sentence_a)
print("B:", sentence_b)
print()
print("мішок слів однаковий:", Counter(words_a) == Counter(words_b))
print()


def ngrams(words, n):
    """Усі послідовності з n слів поспіль. n = 1 — це і є мішок слів."""
    return [tuple(words[i:i + n]) for i in range(len(words) - n + 1)]


print(" n   спільних   різних у A   різних у B")
for n in (1, 2, 3, 4):
    a, b = set(ngrams(words_a, n)), set(ngrams(words_b, n))
    print(f"{n:>2}   {len(a & b):>8}   {len(a - b):>10}   {len(b - a):>11}")

На рівні окремих слів різниці **нуль**: обидва речення дають один і той самиймішок. А означають вони протилежне — одне перетворює асоціативний масив наіндексований, друге навпаки.Уже біграми (пари сусідніх слів) різницю бачать. Це найдешевший спосіб повернутимоделі трохи порядку, і саме ним ми скористаємось у темі[04 «Мішок слів»](../04-bag-of-words/lecture.html). Але біграми — це латка, а нерозвʼязок: пари сусідів не бачать звʼязку між першим і останнім словом речення.## 13 · Чому більше даних не лікуєНайпоширеніша надія: «зберемо більше тексту, і рідкісні слова стануть частими».Перевіримо її прямо. Відкладемо пʼять тисяч документів як тестові, а рештубудемо згодовувати моделі частинами — і дивитись, яка частка тестових слівлишається **невідомою**.Знову три зерна: розбиття випадкове.

In [ ]:
oov_by_seed = {}

t_start = time.time()
for seed in (0, 1, 2):
    order = list(docs)
    random.Random(seed).shuffle(order)
    test_size = min(5000, len(order) // 3)
    test, train = order[:test_size], order[test_size:]

    test_freq = Counter()
    for program, source, target in test:
        test_freq.update(ua_tokens(target))
    test_tokens = sum(test_freq.values())

    marks = grid_upto([1000, 2000, 5000, 10000, 20000, 40000, 60000], len(train))
    seen, i, row = set(), 0, []
    for n in marks:
        while i < n:
            seen.update(ua_tokens(train[i][2]))
            i += 1
        unknown_tokens = sum(c for w, c in test_freq.items() if w not in seen)
        unknown_types = sum(1 for w in test_freq if w not in seen)
        row.append((n, len(seen),
                    100 * unknown_tokens / test_tokens,
                    100 * unknown_types / len(test_freq)))
    oov_by_seed[seed] = row

print(f"тестових документів: {test_size} (числа тестової вибірки — для зерна 2), "
      f"у них {test_tokens} слововживань і {len(test_freq)} словоформ\n")
print("документів у корпусі   словник   невідомих слововживань   невідомих словоформ")
for idx in range(len(oov_by_seed[0])):
    n = oov_by_seed[0][idx][0]
    v = np.mean([oov_by_seed[s][idx][1] for s in (0, 1, 2)])
    tok = [oov_by_seed[s][idx][2] for s in (0, 1, 2)]
    typ = [oov_by_seed[s][idx][3] for s in (0, 1, 2)]
    print(f"{n:>20}   {v:>7.0f}   {np.mean(tok):>21.2f}%   {np.mean(typ):>18.2f}%")
    if idx == len(oov_by_seed[0]) - 1:
        print(f"{'розкид між зернами':>20}   {'':>7}   "
              f"{max(tok)-min(tok):>21.2f}%   {max(typ)-min(typ):>18.2f}%")
print("рахувалось", round(time.time() - t_start, 1), "с")

Тепер найважливіше — не саме число, а **швидкість, з якою воно спадає**.

In [ ]:
n_first = oov_by_seed[0][0][0]
n_last = oov_by_seed[0][-1][0]

alphas = {}
for name, column in (("слововживання", 2), ("словоформи", 3)):
    first = np.mean([oov_by_seed[s][0][column] for s in (0, 1, 2)])
    last = np.mean([oov_by_seed[s][-1][column] for s in (0, 1, 2)])
    # частка невідомих спадає як N у степені мінус alpha — рахуємо цей степінь
    alpha = math.log(first / last) / math.log(n_last / n_first)
    alphas[name] = (alpha, first, last)
    print(f"{name:<14} {first:>6.2f}% -> {last:>5.2f}% при рості корпусу "
          f"у {n_last/n_first:.0f} разів;  спадає як N^-{alpha:.4f};  "
          f"зменшити вдвічі коштує ×{2**(1/alpha):.2f}")

alpha, first, last = alphas["слововживання"]
print()
for goal in (last / 2, last / 5, last / 10):
    factor = (last / goal) ** (1 / alpha)
    print(f"щоб дійти до {goal:.3f}% невідомих слововживань, корпус треба збільшити "
          f"у {factor:>5.1f} раза -> {n_last*factor:>12.0f} документів")

Ось відповідь на «зберемо більше даних». Збереш — і частка невідомих сліввпаде, але **степенево, а не до нуля**. Кожне наступне зменшення вдвічі коштуєприблизно втричі більшого корпусу. Хвіст Ципфа не скорочується від того, що мичитаємо більше: він просто відсувається далі.Саме тому в темі [02](../02-tokenization/lecture.html) курс піде іншим шляхом —не збиратиме більше слів, а **перестане різати текст по словах**.## 14 · Перевірка: наші формули проти бібліотечнихСпека курсу вимагає хоча б одної звірки «наше = бібліотечне». Зробимо дві.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# 1) наш словник проти словника CountVectorizer із тим самим правилом токенізації
sample_docs = [target for _, _, target in docs[:3000]]
vectorizer = CountVectorizer(lowercase=True,
                             token_pattern=r"[абвгґдежзиійклмнопрстуфхцчшщьюяєіїʼ]+")
matrix = vectorizer.fit_transform(sample_docs)

our_vocab = set()
for text in sample_docs:
    our_vocab.update(ua_tokens(text))

assert our_vocab == set(vectorizer.vocabulary_), "словники розійшлися!"
print(f"✅ словник збігається: {len(our_vocab)} словоформ на {len(sample_docs)} документах")
print(f"   матриця CountVectorizer: {matrix.shape[0]} × {matrix.shape[1]}, "
      f"ненульових {matrix.nnz}, заповнено {100*matrix.nnz/(matrix.shape[0]*matrix.shape[1]):.4f}%")

# 2) наша підгонка прямої найменшими квадратами проти np.polyfit
xs = np.log(ranks[:2000]).astype(float)
ys = np.log(values[:2000])
x_mean, y_mean = xs.mean(), ys.mean()
our_slope = ((xs - x_mean) * (ys - y_mean)).sum() / ((xs - x_mean) ** 2).sum()
lib_slope = np.polyfit(xs, ys, 1)[0]

assert np.allclose(our_slope, lib_slope), "нахили розійшлися!"
print(f"✅ нахил збігається: наш {our_slope:.6f}, polyfit {lib_slope:.6f}")

Усередині `polyfit` немає магії: нахил прямої найменших квадратів — це відношенняковаріації до дисперсії, чотири рядки арифметики.І зверни увагу на друге число першої перевірки: матриця «документ × слово» натрьох тисячах документів заповнена **менш ніж на один відсоток**. Це не дрібницяй не привід для оптимізації — це причина, чому мішок слів узагалі можнавикористовувати. Тема [04](../04-bag-of-words/lecture.html) буде саме про це.## 15 · Підсумок числами

In [ ]:
print(f"джерело корпусу               {CORPUS_SOURCE}")
print(f"документів                    {len(docs)} із {len(programs)} програм")
print(f"слововживань (укр)            {len(ua_stream)}")
print(f"словоформ (укр)               {len(ua_freq)}")
print(f"нахил Ципфа (по декадах)      {slope_decades(len(values)):.4f}")
print(f"нахил Ципфа (усі ранги)       {slope_all(len(values)):.4f}")
print(f"словоформ рівно раз           {100*hapax_ua/len(ua_freq):.2f}%")
print(f"слів для 99% тексту           {int(np.searchsorted(cumulative, 0.99) + 1)}")
print(f"показник Гіпса β (укр)        {np.mean(beta_ua):.4f}")
print(f"показник Гіпса β (англ)       {np.mean(beta_en):.4f}")
print(f"показник Гіпса β (генератор)  {np.mean(gen_betas):.4f} "
      f"при словнику {int(np.mean(gen_vocab))} слів")
print(f"невідомих слововживань на повному корпусі {last:.2f}%")

## Завдання### 🟢 Рівень 1 — БазаПовтори замір Гіпса на **англійському** боці корпусу для трьох зерен і побудуйтаблицю «прочитано слів → різних слів» так само, як ми зробили для української.**Зроблено, якщо:** таблиця надрукована, β англійської порахований на трьох зернах,і ти можеш назвати, більший він чи менший за український **і на скільки розкидів**.### 🟡 Рівень 2 — ПлюсВізьми один окремий пакунок із корпусу (наприклад, `docs`, відфільтровані за`program == 'coreutils'`) і поміряй на ньому нахил Ципфа й β. Порівняй із числамивсього корпусу.**Зроблено, якщо:** обидва числа порахувані, і ти письмово пояснив, чому вонивідрізняються від загальнокорпусних — через розмір вибірки чи через вужчий домен.### 🔴 Рівень 3 — ВикликПобудуй **свій** генератор тексту, який має β хоча б 0.3. Підказка: одних лишеслотів зі скінченними списками замало — потрібне джерело нових слів. Виміряйβ на трьох зернах і покажи, що різниця з нулем більша за розкид.**Зроблено, якщо:** β ≥ 0.3 на всіх трьох зернах, і ти назвав словами, яку самевластивість справжнього тексту твій генератор відтворив, а яку ні.## Підказки* Для рівня 2 памʼятай: β залежить від довжини тексту, на якій його міряють.  Порівнюй чесно — обріж великий корпус до довжини маленького.* Для рівня 3 подумай, звідки в справжньому тексті беруться слова, яких раніше  не було: власні назви, числа словами, помилки, складені слова, запозичення.* Якщо словник твого генератора все одно насичується, спробуй додати механізм,  який породжує слово **з частин**, а не бере з готового списку.